<a href="https://colab.research.google.com/github/AdelineKwakye/walmart-huricane-project/blob/main/notebooks/scratch/Amilyn_Estes_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
!git clone https://github.com/AdelineKwakye/walmart-huricane-project.git

fatal: destination path 'walmart-huricane-project' already exists and is not an empty directory.


In [31]:
%cd /content/walmart-huricane-project
!git pull

/content/walmart-huricane-project
Already up to date.


In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

stores = pd.read_csv("data/store_locations.csv")

print("Number of stores:", len(stores))
display(stores.head())

Number of stores: 14


,store_id,store_name,city,state,zip_code,latitude,longitude,region
0,FL-239,Walmart Supercenter Orlando,Orlando,FL,32801,28.5383,-81.3792,Central
1,FL-105,Walmart Supercenter Tampa,Tampa,FL,33602,27.9506,-82.4572,West
2,FL-330,Walmart Supercenter Miami,Miami,FL,33101,25.7617,-80.1918,South
3,FL-412,Walmart Supercenter Jacksonville,Jacksonville,FL,32099,30.3322,-81.6557,North
4,FL-287,Walmart Neighborhood Market Kissimmee,Kissimmee,Florida,34741,28.2920,-81.4076,Central


In [41]:
# Standardize store IDs for weather API requests
valid_stores["store_id"] = (
    valid_stores["store_id"]
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(r"[^A-Z0-9]", "", regex=True)
    .str.replace(r"^FL(\d+)$", r"FL-\1", regex=True)
)

# Keep only one record per store
valid_stores = valid_stores.drop_duplicates(
    subset=["store_id"]
).copy()

print("Unique stores:", len(valid_stores))

display(valid_stores[
    ["store_id", "city", "latitude", "longitude"]
])

# Convert geographic coordinates to numbers
stores["latitude"] = pd.to_numeric(
    stores["latitude"], errors="coerce"
)
stores["longitude"] = pd.to_numeric(
    stores["longitude"], errors="coerce"
)

# Only use locations with valid Florida coordinates
valid_stores = stores[
    stores["latitude"].between(24, 32) &
    stores["longitude"].between(-88, -79)
].copy()

print("Stores with usable coordinates:", len(valid_stores))

display(valid_stores[
    ["store_id", "city", "latitude", "longitude"]
])

Unique stores: 11


,store_id,city,latitude,longitude
0,FL-239,Orlando,28.5383,-81.3792
1,FL-105,Tampa,27.9506,-82.4572
2,FL-330,Miami,25.7617,-80.1918
3,FL-412,Jacksonville,30.3322,-81.6557
4,FL-287,Kissimmee,28.2920,-81.4076
5,FL-156,Daytona Beach,29.2108,-81.0228
6,FL-501,Fort Myers,26.6406,-81.8723
7,FL-078,Gainesville,29.6516,-82.3248
9,FL-445,St. Petersburg,27.7676,-82.6403
10,FL-302,Lakeland,28.0395,-81.9498


Stores with usable coordinates: 13


,store_id,city,latitude,longitude
0,FL-239,Orlando,28.5383,-81.3792
1,FL-105,Tampa,27.9506,-82.4572
2,FL-330,Miami,25.7617,-80.1918
3,FL-412,Jacksonville,30.3322,-81.6557
4,FL-287,Kissimmee,28.2920,-81.4076
5,FL-156,Daytona Beach,29.2108,-81.0228
6,FL-501,Fort Myers,26.6406,-81.8723
7,FL-078,Gainesville,29.6516,-82.3248
9,FL-445,St. Petersburg,27.7676,-82.6403
10,FL-302,Lakeland,28.0395,-81.9498


In [37]:
weather_data = []

for _, store in valid_stores.iterrows():

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": store["latitude"],
        "longitude": store["longitude"],
        "start_date": "2024-09-01",
        "end_date": "2024-09-30",
        "hourly": "wind_speed_10m,precipitation,pressure_msl",
        "wind_speed_unit": "mph",
        "timezone": "America/New_York"
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()

        hourly = pd.DataFrame(response.json()["hourly"])

        # Convert hourly observations to daily dates
        hourly["date"] = pd.to_datetime(
            hourly["time"]
        ).dt.normalize()

        # Summarize weather by day
        daily = hourly.groupby("date").agg(
            wind_speed=("wind_speed_10m", "max"),
            precipitation=("precipitation", "sum"),
            pressure=("pressure_msl", "min")
        ).reset_index()

        daily["store_id"] = store["store_id"]
        daily["city"] = store["city"]

        weather_data.append(daily)

        print("Completed:", store["city"])

    except requests.RequestException as e:
        print("Failed:", store["city"], e)

if not weather_data:
    raise ValueError("No weather data was retrieved.")

weather_df = pd.concat(weather_data, ignore_index=True)

display(weather_df.head())


Completed: Orlando
Completed: Tampa
Completed: Miami
Completed: Jacksonville
Completed: Kissimmee
Completed: Daytona Beach
Completed: Fort Myers
Completed: Gainesville
Completed: St. Petersburg
Completed: Lakeland
Completed: Pensacola
Completed: Orlando
Completed: Kissimmee


,date,wind_speed,precipitation,pressure,store_id,city
0,2024-09-01,11.6,6.4,1012.8,FL-239,Orlando
1,2024-09-02,7.9,22.0,1013.0,FL-239,Orlando
2,2024-09-03,12.8,32.7,1014.3,FL-239,Orlando
3,2024-09-04,7.8,21.1,1016.5,FL-239,Orlando
4,2024-09-05,9.4,20.7,1014.8,FL-239,Orlando
